# Week 11 guided practice

CML3112. Follow the lecturer’s PowerPoint in this order. This notebook contains worked examples, not your graded upload.

Extract the student ZIP and upload engineering_data.csv into Colab with the Files panel before running the cells. The target in this notebook is cooling load in kW.

## Today’s learning

*PowerPoint slide 2*

WHAT WE WILL DO
Build new input columns from the ones we have, test whether each one helps, and read which input moves a prediction more. Learn the words feature engineering, interaction and ablation.

HOW YOU WILL SHOW IT
You can build a derived feature, compare models with and without it on the same rows, and decide from the evidence whether to keep it.

## The supplied file

*PowerPoint slide 4*

Extract this week’s student ZIP. Upload engineering_data.csv into Colab using the Files panel, or keep them beside the notebook locally.

The examples below read this file. Run the cells in slide order. Keep the original CSV unchanged.

## Loading the supplied observations

*PowerPoint slide 5*

raw_data: original CSV table; keep it unchanged so the analysis can be checked.

Run the cell and read the output below it.

In [ ]:
import pandas as pd
raw_data = pd.read_csv("engineering_data.csv")
print(raw_data.head())
print(raw_data.shape)
print(raw_data.dtypes)

## Preparing the supplied modelling rows: step 1

*PowerPoint slide 6*

modelling_data: copied observations after duplicate and validity checks.

Run the cell and read the output below it.

In [ ]:
modelling_data = raw_data.drop_duplicates("record_id").copy()
target_is_recorded = modelling_data["target"].notna()
load_is_valid = modelling_data["load_pct"].between(0, 100)
rows_to_keep = target_is_recorded & load_is_valid
print(target_is_recorded.head())
print(load_is_valid.head())
modelling_data = modelling_data[rows_to_keep].copy()
print(len(raw_data), len(modelling_data))

## Choose groups before choosing rows

*PowerPoint slide 7*

isin(list_of_labels) answers True when a row’s equipment ID appears in the list.

Here E1–E4 supply training rows, E5 validation rows and E6 test rows.

The groups stay separate so one equipment group cannot appear on both sides.

## Preparing the supplied modelling rows: step 2

*PowerPoint slide 8*

modelling_data: copied observations after duplicate and validity checks.

Run the cell and read the output below it.

In [ ]:
is_training_equipment = modelling_data["equipment_id"].isin(
    ["E1", "E2", "E3", "E4"]
)
is_validation_equipment = modelling_data["equipment_id"].isin(
    ["E5"]
)
is_test_equipment = modelling_data["equipment_id"].isin(["E6"])

## Preparing the supplied modelling rows: step 3

*PowerPoint slide 9*

training_data: rows used to learn model settings; only these rows may determine imputation values.

validation_data: separate rows used to compare models without fitting on them.

## Preparing the supplied modelling rows: step 3

*PowerPoint slide 10*

modelling_data: copied observations after duplicate and validity checks.

Run the cell and read the output below it.

In [ ]:
training_data = modelling_data[is_training_equipment].copy()
validation_data = modelling_data[is_validation_equipment].copy()
test_data = modelling_data[is_test_equipment].copy()
feature_columns = ["load_pct", "ambient_c"]
print(len(training_data), len(validation_data), len(test_data))

## A physical relationship can form a feature

*PowerPoint slide 11*

Explain each quantity in the relationship and its unit before reading the pattern.

## A straight line misses a curve, and a new feature repairs it

*PowerPoint slide 12*

Invented data. Adding x squared as a feature lets the same linear model follow the curve.

## Derived features from existing inputs

*PowerPoint slide 13*

load_pct / 100 gives load_fraction. Multiplying load_pct by ambient_c gives the load_ambient interaction.

equipment_data["new_name"] = expression adds a column. Apply the same formulas to training and validation rows, without changing raw data.

## Operations for the next example

*PowerPoint slide 14*

Adding or replacing a column: equipment_data["name"] = values assigns that column. An array follows row order. A Series aligns by row label, so keep labels paired with the intended observations.

Column arithmetic: Arithmetic on columns calculates one result per row. Series align by row label. Divide only where the denominator is valid and keep actual and predicted observations paired.

## Defining and calling a function

*PowerPoint slide 15*

def gives a reusable block a name and input parameters. The indented statements run when it is called. return sends a result back to the caller; print displays a result but does not return it.

## Features from the supplied load and ambient columns

*PowerPoint slide 17*

Use the inputs and operations defined in the preceding slides.

Run the cell and read the output below it.

In [ ]:
def add_features(equipment_data):
    extended_data = equipment_data.copy()
    extended_data["load_fraction"] = (
        extended_data["load_pct"] / 100
    )
    extended_data["load_ambient"] = (
        extended_data["load_pct"] * extended_data["ambient_c"]
    )
    return extended_data
training_features = add_features(training_data)
validation_features = add_features(validation_data)
print(
    training_features[
        ["load_pct", "load_fraction", "load_ambient"]
    ].head()
)

## Which input moves the prediction more?

*PowerPoint slide 18*

effect of one spread = slope × standard deviation
Slopes come from the linear model fitted on the training rows. Spreads are the population standard deviation of the same rows, as StandardScaler calculates it.

## Ablation compares the same observations

*PowerPoint slide 19*

Fit one linear model using load_pct and ambient_c. Fit a second using those columns plus load_ambient.

Compare both on the same validation targets. Improvement equals baseline MAE minus extended MAE.

## Change one feature set at a time

*PowerPoint slide 20*

The base inputs are load and ambient temperature.

The extended inputs also include their product.

Train separate pipelines on the same rows and compare on the same validation rows.

This isolates the effect of adding the feature.

## Paired ablation using the supplied CSV: step 1

*PowerPoint slide 21*

base_feature_columns: input columns for the simpler comparison pipeline.

Run the cell and read the output below it.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
base_feature_columns = ["load_pct", "ambient_c"]
extended_feature_columns = base_feature_columns + ["load_ambient"]

## Fit both alternatives fairly

*PowerPoint slide 22*

Each pipeline learns its own training medians and regression settings.

The base model sees only base columns; the extended model sees the additional feature.

Neither uses validation targets while learning.

## Paired ablation using the supplied CSV: step 2

*PowerPoint slide 24*

base_feature_columns: input columns for the simpler comparison pipeline.

Run the cell and read the output below it.

In [ ]:
base_pipeline = make_pipeline(
    SimpleImputer(strategy="median"), LinearRegression()
)
extended_pipeline = make_pipeline(
    SimpleImputer(strategy="median"), LinearRegression()
)
base_pipeline.fit(
    training_features[base_feature_columns],
    training_data["target"],
)
extended_pipeline.fit(
    training_features[extended_feature_columns],
    training_data["target"],
)

## Compare errors on the same rows

*PowerPoint slide 25*

Predict both alternatives on matching validation rows. mean_absolute_error compares each prediction with its observed target.

Base MAE minus extended MAE is positive when the extra feature helped on this validation set.

## Paired ablation using the supplied CSV: step 3

*PowerPoint slide 26*

base_feature_columns: input columns for the simpler comparison pipeline.

Run the cell and read the output below it.

In [ ]:
base_predictions = base_pipeline.predict(
    validation_features[base_feature_columns]
)
base_absolute_error = mean_absolute_error(
    validation_data["target"], base_predictions
)

## Paired ablation using the supplied CSV: step 4

*PowerPoint slide 27*

extended_predictions: validation predictions from the extended pipeline.

extended_absolute_error: validation mean absolute error for the extended pipeline.

## Paired ablation using the supplied CSV: step 4

*PowerPoint slide 29*

base_feature_columns: input columns for the simpler comparison pipeline.

Run the cell and read the output below it.

In [ ]:
extended_predictions = extended_pipeline.predict(
    validation_features[extended_feature_columns]
)
extended_absolute_error = mean_absolute_error(
    validation_data["target"], extended_predictions
)
print(
    round(base_absolute_error, 4),
    round(extended_absolute_error, 4),
)
print(round(base_absolute_error - extended_absolute_error, 4))

## A feature earns its place only if it helps

*PowerPoint slide 30*

Base error 0.7427 kW. With load × ambient 0.7476 kW.

The extra feature changed the validation error by +0.0050 kW. The error went up.
A derived feature inherits the blanks of the columns it is built from. One training row has a blank ambient temperature, so its product is blank and the imputer fills it with a typical value.
With only 15 validation rows the comparison is rough. A feature that adds a column and gives no clear gain should be dropped. Keep the simpler model.

## Use only inputs you will have at prediction time

*PowerPoint slide 31*

ALLOWED
Load now. Ambient temperature now. A value calculated from those two, such as load divided by 100.
Anything the sensor could report at the moment the prediction is requested.

LEAKS THE ANSWER
The measured target itself. A repair note written after the fault. A monthly average that includes next week.
A feature like these makes training look wonderful and fails in service.

## Your issued task

*PowerPoint slide 32*

01

Investigate building cooling load.

02

Target units: kW.

03

These are synthetic teaching records for practice.

04

Each record is one operating observation; equipment_id repeats across observations.

## make_features(feature_records)

*PowerPoint slide 33*

Return one feature row per input, in order: [load_pct, ambient_c, load_pct/100, load_pct*ambient_c].

Use inputs available at prediction time only.

Keep the exact names and outputs. Check types, units and the issued brief.

## ablation_metrics(training_records, validation_records)

*PowerPoint slide 34*

Fit two sklearn LinearRegression models on the same training records: base features [load_pct, ambient_c] and extended features adding load_pct*ambient_c.

Evaluate both on the same validation records.

Return base_mae, extended_mae and improvement=base_mae-extended_mae, without rounding.

Inputs are complete; never fit on validation targets.

Keep the exact names and outputs. Check types, units and the issued brief.

## Written evidence

*PowerPoint slide 35*

01

Use the cell headed “My written evidence”.

02

Explain the supplied scenario in your own words.

03

State target units, reasons and limitations.

04

Submit one complete notebook containing code and written evidence.